In [ ]:
!pip install -U tensorflow

import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from sklearn.manifold import TSNE

def load_and_preprocess_image(image_path, target_size=(224, 224)):
    try:
        img = cv2.imread(image_path)

        if img is None:
            return None

        img = cv2.resize(img, target_size)
        img = img / 255.0

        return img

    except Exception as e:
        print(f"Error loading or processing image {image_path}: {e}")
        return None

preprocessed_images = []
image_labels = []

image_extensions = ['.jpg', '.jpeg', '.png']
subset_size_per_class = 100

for subdirectory in subdirectories:
    subdirectory_path = os.path.join(DATA_ROOT, subdirectory)

    if os.path.isdir(subdirectory_path):
        images_in_subdir = [
            f for f in os.listdir(subdirectory_path)
            if os.path.splitext(f)[1].lower() in image_extensions
        ]

        subset_images = images_in_subdir[:subset_size_per_class]

        for filename in subset_images:
            image_path = os.path.join(subdirectory_path, filename)

            preprocessed_img = load_and_preprocess_image(image_path)

            if preprocessed_img is not None:
                preprocessed_images.append(preprocessed_img)
                image_labels.append(subdirectory)

preprocessed_images = np.array(preprocessed_images)
image_labels = np.array(image_labels)

print(f"Loaded and preprocessed {len(preprocessed_images)} images.")
print(f"Shape of preprocessed_images array: {preprocessed_images.shape}")
print(f"Shape of image_labels array: {image_labels.shape}")

base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

feature_extractor = Model(
    inputs=base_model.input,
    outputs=base_model.output
)

image_embeddings = feature_extractor.predict(preprocessed_images)

print(f"Shape of generated image embeddings: {image_embeddings.shape}")

num_images, height, width, channels = image_embeddings.shape

image_embeddings_reshaped = image_embeddings.reshape(
    num_images,
    height * width * channels
)

tsne = TSNE(
    n_components=2,
    random_state=42,
    perplexity=30,
    n_iter=300
)

embeddings_reduced = tsne.fit_transform(image_embeddings_reshaped)

print(f"Shape of reduced embeddings: {embeddings_reduced.shape}")

plt.figure(figsize=(10, 8))

sns.scatterplot(
    x=embeddings_reduced[:, 0],
    y=embeddings_reduced[:, 1],
    hue=image_labels,
    palette='viridis',
    legend='full'
)

plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')

plt.grid(True)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

labels = np.asarray(image_labels)

if labels.dtype.kind not in "iu":
    unique = np.unique(labels)

    label_map = {
        c: i for i, c in enumerate(unique)
    }

    y = np.array(
        [label_map[v] for v in labels],
        dtype=int
    )

else:
    y = labels.astype(int)

X = StandardScaler().fit_transform(image_embeddings_reshaped)

X_eval, y_eval = X, y

classes, counts = np.unique(y_eval, return_counts=True)

if len(classes) < 2:
    raise ValueError(
        "Need at least 2 classes to compute separability metrics."
    )

if np.any(counts < 2):
    raise ValueError(
        "Silhouette score requires each class to have at least 2 samples in the evaluated set."
    )

sil = silhouette_score(
    X_eval,
    y_eval,
    metric="euclidean"
)

dbi = davies_bouldin_score(
    X_eval,
    y_eval
)

chi = calinski_harabasz_score(
    X_eval,
    y_eval
)

print("=== Separability metrics on VGG16 embeddings ===")
print(f"Silhouette score (higher=better): {sil:.4f}")
print(f"Davies–Bouldin index (lower=better): {dbi:.4f}")
print(f"Calinski–Harabasz index (higher=better, optional): {chi:.2f}")

sil_tsne = silhouette_score(
    embeddings_reduced,
    y_eval,
    metric="euclidean"
)

dbi_tsne = davies_bouldin_score(
    embeddings_reduced,
    y_eval
)

chi_tsne = calinski_harabasz_score(
    embeddings_reduced,
    y_eval
)

print("\n=== Separability metrics on 2D t-SNE projection (optional) ===")
print(f"Silhouette score (2D): {sil_tsne:.4f}")
print(f"Davies–Bouldin index (2D): {dbi_tsne:.4f}")
print(f"Calinski–Harabasz index (2D, optional): {chi_tsne:.2f}")

silhouette_tsne = sil_tsne
davies_bouldin_tsne = dbi_tsne

metrics = [
    'Silhouette Score',
    'Davies–Bouldin Index'
]

values = [
    silhouette_tsne,
    davies_bouldin_tsne
]

plt.figure(figsize=(6, 4))

bars = plt.bar(
    metrics,
    values
)

for bar in bars:
    height = bar.get_height()

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f'{height:.2f}',
        ha='center',
        va='bottom'
    )

plt.ylabel('Metric Value')

plt.grid(
    axis='y',
    linestyle='--',
    alpha=0.6
)

plt.tight_layout()

plt.show()

In [ ]:
from sklearn.metrics import silhouette_samples
import numpy as np

silhouette_vals = silhouette_samples(embeddings_reduced, y_eval)

plt.figure(figsize=(7, 5))
y_lower = 10

for cls in np.unique(y_eval):
    cls_vals = silhouette_vals[y_eval == cls]
    cls_vals.sort()
    y_upper = y_lower + len(cls_vals)

    plt.fill_betweenx(
        np.arange(y_lower, y_upper),
        0,
        cls_vals,
        alpha=0.7,
        label=f'Class {cls}'
    )

    y_lower = y_upper + 10

plt.axvline(silhouette_tsne, linestyle='--', linewidth=2, label='Mean Silhouette')
plt.xlabel('Silhouette Coefficient Value')
plt.ylabel('Samples')

plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist


X = StandardScaler().fit_transform(image_embeddings_reshaped)
y = y_eval

class_labels = np.unique(y)
class_names = ['Normal', 'Bacterial', 'Viral']


centroids = np.array([X[y == cls].mean(axis=0) for cls in class_labels])


distance_matrix = cdist(centroids, centroids, metric='euclidean')

plt.figure(figsize=(6, 5))
sns.heatmap(
    distance_matrix,
    annot=True,
    fmt=".2f",
    xticklabels=class_names,
    yticklabels=class_names,
    cmap="viridis"
)

plt.tight_layout()
plt.show()
